# ITS Predict

In [ ]:
%pip install -qq --ignore-requires-python --no-deps 'graphies[predict] @ git+https://github.com/lukasmki/graphies.git'
%pip install -qq pydantic networkx datasets polars torch

import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
else:
    print(
        "CUDA is not available. Please ensure you have selected a GPU runtime in 'Runtime > Change runtime type'."
    )

## Setup Data Sources

In [ ]:
!mkdir -p its-data
!wget -nv https://raw.githubusercontent.com/lukasmki/graphies-applications/refs/heads/main/grammars/its.json -O its-data/its.json

from pathlib import Path

GRAMMAR_PATH = next(Path().glob("its-data/*.json"))

In [ ]:
from datasets import load_dataset

hf_dataset = load_dataset("lukaskim/MappedCRD", "targets", split="train")

## Setup Data Loaders

In [ ]:
from graphies.predict import GraphiesTokenizer, HFGraphiesDataset
from torch.utils.data import DataLoader, random_split

tokenizer = GraphiesTokenizer(GRAMMAR_PATH)
dataset = HFGraphiesDataset(
    hf_dataset, column="its_graphies", tokenizer=tokenizer, split=None
)

trn, tst = random_split(dataset, [0.9, 0.1])
torch.save(
    {"trn_indices": trn.indices, "tst_indices": tst.indices}, "its-data/split.pt"
)
trn_loader = DataLoader(
    dataset=trn,
    batch_size=256,
    shuffle=True,
    collate_fn=tokenizer.collate,
)
tst_loader = DataLoader(
    dataset=tst,
    batch_size=256,
    shuffle=False,
    collate_fn=tokenizer.collate,
)

## Trainer

In [ ]:
from graphies.predict import GraphiesTrainer
from graphies.predict.models import GRU
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GRU(vocab_size=tokenizer.vocab_size)
optimizer = Adam(params=model.parameters(), lr=1e-3)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.1)

# include kwargs for classes to restart from checkpoitn
checkpoint = {
    "model_kwargs": {"vocab_size": tokenizer.vocab_size},
    "optimizer_kwargs": {"lr": 1e-3},
    "scheduler_kwargs": {"mode": "min", "patience": 3, "factor": 0.1},
}
trainer = GraphiesTrainer(model, optimizer, scheduler, device, checkpoint)

In [ ]:
trainer.train(
    train=trn_loader,
    epochs=50,
    test=tst_loader,
    test_interval=1,
    log="its-data/log.csv",
    log_interval=1,
    checkpoint="its-data/chk.pt",
    checkpoint_interval=1,
)
trainer.save_model("its-data/model.pt")

In [ ]:
# export to google drive
from google.colab import drive

drive.mount("/content/drive")
!zip -r its-data.zip its-data/
!cp its-data.zip '/content/drive/MyDrive/'

# Run Inference

In [ ]:
from graphies.predict import GraphiesModel

model = GraphiesModel.from_checkpoint(
    checkpoint="its-data/model.pt",
    tokenizer=tokenizer,
    model_cls=GRU,
    device=device,
)
sequences = model.generate(num=1024, temperature=0.9, top_p=0.95, max_len=5000)
ref_sequences = hf_dataset[tst.indices]["graphies"]